# Yam Yabasha — full pipeline and parameter playground

This notebook explains and visualizes the exact production algorithm in `src/yam_yabasha`.

It is intentionally split into two parts:

1. **Expensive stage** — registration and CLIPSeg prompt maps. Run this once per image.
2. **Fast stage** — ROI, thresholding, morphology, contour scoring, and smoothing. Rerun this as often as needed while tuning parameters.

All computation stays at full resolution. Display copies are reduced only for plotting, so the notebook remains responsive.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

from IPython import get_ipython
from IPython.display import display
get_ipython().run_line_magic("matplotlib", "inline")

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / "yam-yabasha", cwd.parent / "yam-yabasha"]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find the yam-yabasha project root")


ROOT = find_project_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from yam_yabasha.config import PipelineConfig, ROIContourConfig, SegmentationConfig
from yam_yabasha.io import list_images, load_hint, load_image
from yam_yabasha.pipeline import ShorelinePipeline
from yam_yabasha.registration import warp_image
from yam_yabasha.shoreline import extract_shoreline
from yam_yabasha.visualization import draw_polyline, overlay_mask, registration_overlay

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11})


def preview(image, max_side=1100):
    """Downsize only the displayed copy; processing remains full-resolution."""
    height, width = image.shape[:2]
    scale = min(1.0, max_side / max(height, width))
    if scale == 1.0:
        return image
    return cv2.resize(
        image,
        (int(round(width * scale)), int(round(height * scale))),
        interpolation=cv2.INTER_AREA,
    )


def rgb(image_bgr):
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


print(f"Project root: {ROOT}")

## 1. Experiment controls

Start with `USE_REFERENCE = True` to inspect a tracked camera reference without registration. Set it to `False` and choose `IMAGE_INDEX` to inspect a registered test frame. The README figures are generated separately from labeled examples by `scripts/generate_readme_assets.py`.

Changing prompts requires rerunning CLIPSeg. Changing contour parameters later does **not**.

In [ ]:
# ---------- Image selection ----------
LOCATION = "entrance"          # "entrance" or "porch"
USE_REFERENCE = True           # False selects a frame from Test/<location>
IMAGE_INDEX = 0                # used only when USE_REFERENCE=False

# ---------- Expensive CLIPSeg settings ----------
WATER_PROMPTS = (
    "water",
    "sea water",
    "foamy water",
    "water with foam",
    "waves on water",
)
LAND_PROMPTS = (
    "sand",
    "beach",
    "wet sand",
    "rocky shore",
    "land",
    "coast",
)
WATER_COMBINE = "max"          # "max" or "mean"
LAND_COMBINE = "max"           # "max" or "mean"
DEVICE = None                  # None = automatic, or "cpu" / "cuda"

# ---------- Fast contour settings ----------
CONTOUR_CONFIG = ROIContourConfig(
    corridor_radius_px=120,   # validated global default
    contrast_threshold=-0.05, # validated global default
    blur_sigma=2.0,
    morphology_kernel=5,
    close_iterations=2,
    open_iterations=1,
    semantic_radius_px=7,
    min_run_points=20,
    min_run_length_px=50,
    length_weight=0.25,
    smoothing_window=7,
    border_margin_px=2,
)

location_dir = ROOT / "data" / LOCATION
reference_path = location_dir / "reference.jpg"
if USE_REFERENCE:
    image_path = reference_path
    register_image = False
else:
    available_images = list_images(ROOT / "Test" / LOCATION)
    if not available_images:
        raise FileNotFoundError(f"No test images found for {LOCATION}")
    image_path = available_images[IMAGE_INDEX]
    register_image = True

print(f"Location: {LOCATION}")
print(f"Image: {image_path.name}")
print(f"Registration enabled: {register_image}")

## 2. Run registration and CLIPSeg once

This is the expensive cell. Rerun it after changing the image, location, prompts, prompt-combination mode, or device.

In [ ]:
segmentation_config = SegmentationConfig(
    water_prompts=WATER_PROMPTS,
    land_prompts=LAND_PROMPTS,
    water_combine=WATER_COMBINE,
    land_combine=LAND_COMBINE,
    device=DEVICE,
)
config = PipelineConfig(
    project_root=ROOT,
    segmentation=segmentation_config,
    contour=CONTOUR_CONFIG,
)
pipeline = ShorelinePipeline(config)
result = pipeline.process(
    image_path,
    LOCATION,
    register=register_image,
    save_outputs=False,
)
seg = result.segmentation
hint_points = load_hint(location_dir / "shoreline_hint.json")

print(f"Aligned image shape: {result.aligned_image.shape}")
print(f"Water prompts: {len(seg.water_prompt_maps)}")
print(f"Land prompts: {len(seg.land_prompt_maps)}")
if result.registration is not None:
    print("Registration metrics:")
    for key, value in result.registration.metrics.items():
        print(f"  {key}: {value}")

## 3. Registration inspection

For a test frame, the red/cyan overlay makes registration errors visible: aligned static structures should overlap. Reference mode skips registration because the input already is the reference. The README registration figure is generated from a deliberately displaced real test frame.

In [ ]:
reference_bgr = load_image(reference_path)
input_bgr = load_image(image_path)
registration_check = registration_overlay(reference_bgr, result.aligned_image)

panels = [
    ("Reference", rgb(reference_bgr)),
    ("Input frame", rgb(input_bgr)),
    ("Aligned frame", rgb(result.aligned_image)),
    ("Registration overlay", rgb(registration_check)),
]
figure, axes = plt.subplots(1, 4, figsize=(18, 5))
for axis, (title, image) in zip(axes, panels):
    axis.imshow(preview(image))
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()
plt.show()
plt.close(figure)

## 4. Every text prompt separately

These maps reveal which words help and which words create false detections. A poor final shoreline often starts with one overly broad prompt dominating a `max` ensemble.

In [ ]:
def plot_prompt_maps(prompt_maps, group_name, columns=3):
    items = list(prompt_maps.items())
    rows = int(np.ceil(len(items) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(5 * columns, 4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, (prompt, probability) in zip(axes, items):
        handle = axis.imshow(preview(probability), cmap="turbo", vmin=0, vmax=1)
        axis.set_title(f'{group_name}: "{prompt}"')
        axis.axis("off")
        figure.colorbar(handle, ax=axis, fraction=0.046, pad=0.02)
    for axis in axes[len(items):]:
        axis.axis("off")
    figure.tight_layout()
    plt.show()
    plt.close(figure)


plot_prompt_maps(seg.water_prompt_maps, "Water")
plot_prompt_maps(seg.land_prompt_maps, "Land")

## 5. Combined semantic maps

`Pwater` and `Pland` are combined independently. The signed map `Pwater - Pland` is the actual semantic decision surface used by the contour stage.

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(preview(rgb(result.aligned_image)))
axes[0].set_title("Aligned image")

water_handle = axes[1].imshow(preview(seg.water_probability), cmap="turbo", vmin=0, vmax=1)
axes[1].set_title(f"Combined water ({WATER_COMBINE})")
figure.colorbar(water_handle, ax=axes[1], fraction=0.046)

land_handle = axes[2].imshow(preview(seg.land_probability), cmap="turbo", vmin=0, vmax=1)
axes[2].set_title(f"Combined land ({LAND_COMBINE})")
figure.colorbar(land_handle, ax=axes[2], fraction=0.046)

contrast_handle = axes[3].imshow(preview(seg.contrast), cmap="coolwarm", vmin=-1, vmax=1)
axes[3].set_title("Contrast = Pwater - Pland")
figure.colorbar(contrast_handle, ax=axes[3], fraction=0.046)

for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()
plt.close(figure)

## 6. Fast contour extraction

Edit `CONTOUR_CONFIG` in the controls cell or replace selected values below, then rerun from this cell onward.

Most influential parameters:

- `corridor_radius_px`: how far from the user hint a line may be considered.
- `contrast_threshold`: how strongly water must beat land.
- `blur_sigma`: semantic-map smoothing before thresholding.
- `close_iterations` / `open_iterations`: connect gaps or remove fragments.
- `semantic_radius_px`: neighborhood requiring nearby support from both classes.
- `min_run_length_px`: rejects short candidate segments.
- `length_weight`: preference for long candidates versus strong semantic evidence.
- `smoothing_window`: final polyline smoothing only.

In [ ]:
# Optional quick overrides: uncomment values you want to test.
fast_config = replace(
    CONTOUR_CONFIG,
    # corridor_radius_px=140,
    # contrast_threshold=0.05,
    # blur_sigma=3.0,
    # close_iterations=3,
    # open_iterations=0,
    # semantic_radius_px=10,
    # min_run_length_px=80,
    # length_weight=0.15,
    # smoothing_window=11,
)

shoreline = extract_shoreline(
    seg.water_probability,
    seg.land_probability,
    hint_points,
    fast_config,
)
print(f"Candidates: {len(shoreline.candidates)}")
print(f"Selected points: {len(shoreline.points)}")
print(f"Confidence: {shoreline.confidence:.4f}")

## 7. Every contour-pipeline stage

The contour is extracted from the full water region first and clipped to the yellow corridor only afterwards. This prevents the corridor border itself from becoming a false shoreline.

In [ ]:
corridor_overlay = overlay_mask(result.aligned_image, shoreline.corridor_mask)
cv2.polylines(
    corridor_overlay,
    [np.round(hint_points).astype(np.int32)],
    False,
    (255, 0, 255),
    4,
)

all_candidates_overlay = result.aligned_image.copy()
candidate_colors = [(0, 255, 255), (255, 0, 255), (255, 255, 0), (0, 165, 255)]
for index, candidate in enumerate(shoreline.candidates[:10]):
    color = candidate_colors[index % len(candidate_colors)]
    cv2.polylines(all_candidates_overlay, [candidate.points], False, color, 2)

selected_overlay = draw_polyline(result.aligned_image, shoreline.points, (0, 0, 255), 4)

stages = [
    ("1. Aligned image", rgb(result.aligned_image), None, None),
    ("2. Hint + search corridor", rgb(corridor_overlay), None, None),
    ("3. Raw contrast", shoreline.contrast, "coolwarm", (-1, 1)),
    ("4. Smoothed contrast", shoreline.smooth_contrast, "coolwarm", (-1, 1)),
    ("5. Thresholded water region", shoreline.water_region, "gray", (0, 255)),
    ("6. Contrast gradient", shoreline.gradient, "magma", (0, 1)),
    ("7. Nearby water AND land", shoreline.semantic_support, "viridis", (0, 1)),
    ("8. Final boundary score", shoreline.boundary_score, "magma", (0, 1)),
    ("9. Top contour candidates", rgb(all_candidates_overlay), None, None),
    ("10. Selected shoreline", rgb(selected_overlay), None, None),
]

figure, axes = plt.subplots(2, 5, figsize=(21, 9))
for axis, (title, image, cmap, limits) in zip(axes.ravel(), stages):
    kwargs = {"cmap": cmap, "interpolation": "nearest"}
    if limits is not None:
        kwargs.update(vmin=limits[0], vmax=limits[1])
    axis.imshow(preview(image), **kwargs)
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()
plt.show()
plt.close(figure)

## 8. Candidate ranking

Inspect whether the selected line won because it had strong semantic evidence or merely because it was long. Candidate 0 is the selected candidate.

In [ ]:
candidate_table = pd.DataFrame(
    [
        {
            "rank": index,
            "score": candidate.score,
            "mean_boundary_evidence": candidate.mean_evidence,
            "length_px": candidate.length_px,
            "normalized_length": candidate.length_fraction,
            "points": len(candidate.points),
        }
        for index, candidate in enumerate(shoreline.candidates[:20])
    ]
)
candidate_table.round(4)

## 9. Fast parameter sweeps

These cells reuse the saved CLIPSeg maps. They change one parameter at a time, which makes the reason for a different shoreline much easier to diagnose.

In [ ]:
THRESHOLDS = [-0.10, -0.05, 0.00, 0.05, 0.10]

figure, axes = plt.subplots(1, len(THRESHOLDS), figsize=(20, 5))
threshold_results = {}
for axis, threshold in zip(axes, THRESHOLDS):
    sweep_config = replace(fast_config, contrast_threshold=threshold)
    sweep_result = extract_shoreline(
        seg.water_probability,
        seg.land_probability,
        hint_points,
        sweep_config,
    )
    threshold_results[threshold] = sweep_result
    overlay = draw_polyline(result.aligned_image, sweep_result.points, (0, 0, 255), 4)
    axis.imshow(preview(rgb(overlay)))
    axis.set_title(
        f"threshold={threshold:+.2f}\n"
        f"candidates={len(sweep_result.candidates)} | points={len(sweep_result.points)}"
    )
    axis.axis("off")
figure.suptitle("Contrast-threshold sensitivity")
figure.tight_layout()
plt.show()
plt.close(figure)

In [ ]:
CORRIDOR_RADII = [50, 75, 100, 140, 180]

figure, axes = plt.subplots(1, len(CORRIDOR_RADII), figsize=(20, 5))
radius_results = {}
for axis, radius in zip(axes, CORRIDOR_RADII):
    sweep_config = replace(fast_config, corridor_radius_px=radius)
    sweep_result = extract_shoreline(
        seg.water_probability,
        seg.land_probability,
        hint_points,
        sweep_config,
    )
    radius_results[radius] = sweep_result
    corridor_view = overlay_mask(result.aligned_image, sweep_result.corridor_mask)
    overlay = draw_polyline(corridor_view, sweep_result.points, (0, 0, 255), 4)
    axis.imshow(preview(rgb(overlay)))
    axis.set_title(
        f"radius={radius}px\n"
        f"candidates={len(sweep_result.candidates)} | points={len(sweep_result.points)}"
    )
    axis.axis("off")
figure.suptitle("Search-corridor sensitivity")
figure.tight_layout()
plt.show()
plt.close(figure)

In [ ]:
EXPERIMENTS = {
    "baseline": fast_config,
    "stricter semantics": replace(fast_config, contrast_threshold=0.08),
    "more connected mask": replace(fast_config, close_iterations=4, open_iterations=0),
    "prefer evidence": replace(fast_config, length_weight=0.05),
    "prefer long contour": replace(fast_config, length_weight=0.50),
    "strong smoothing": replace(fast_config, blur_sigma=4.0, smoothing_window=15),
}

columns = 3
rows = int(np.ceil(len(EXPERIMENTS) / columns))
figure, axes = plt.subplots(rows, columns, figsize=(16, 5 * rows))
axes = np.atleast_1d(axes).ravel()
experiment_results = {}
for axis, (name, experiment_config) in zip(axes, EXPERIMENTS.items()):
    experiment_result = extract_shoreline(
        seg.water_probability,
        seg.land_probability,
        hint_points,
        experiment_config,
    )
    experiment_results[name] = experiment_result
    overlay = draw_polyline(result.aligned_image, experiment_result.points, (0, 0, 255), 4)
    axis.imshow(preview(rgb(overlay)))
    axis.set_title(
        f"{name}\nscore={experiment_result.confidence:.3f} | "
        f"candidates={len(experiment_result.candidates)}"
    )
    axis.axis("off")
for axis in axes[len(EXPERIMENTS):]:
    axis.axis("off")
figure.suptitle("Editable contour-configuration comparison")
figure.tight_layout()
plt.show()
plt.close(figure)

## 10. Optional multi-image preview

Set `RUN_BATCH_PREVIEW = True` to inspect the same configuration across several dates. The already loaded model is reused. This is slower because CLIPSeg still has to process every image.

In [ ]:
RUN_BATCH_PREVIEW = False  # requires local Test/<location> images
MAX_BATCH_IMAGES = 6

if RUN_BATCH_PREVIEW:
    batch_images = list_images(ROOT / "Test" / LOCATION)[:MAX_BATCH_IMAGES]
    batch_outputs = []
    batch_failures = []
    for batch_image_path in batch_images:
        try:
            batch_result = pipeline.process(
                batch_image_path,
                LOCATION,
                register=True,
                save_outputs=False,
            )
            batch_shoreline = extract_shoreline(
                batch_result.segmentation.water_probability,
                batch_result.segmentation.land_probability,
                hint_points,
                fast_config,
            )
            batch_outputs.append((batch_image_path, batch_result, batch_shoreline))
        except Exception as error:
            batch_failures.append((batch_image_path.name, str(error)))

    columns = 3
    rows = max(1, int(np.ceil(len(batch_outputs) / columns)))
    figure, axes = plt.subplots(rows, columns, figsize=(16, 5 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, (path, batch_result, batch_shoreline) in zip(axes, batch_outputs):
        overlay = draw_polyline(
            batch_result.aligned_image,
            batch_shoreline.points,
            (0, 0, 255),
            4,
        )
        axis.imshow(preview(rgb(overlay)))
        axis.set_title(
            f"{path.name}\n"
            f"candidates={len(batch_shoreline.candidates)} | "
            f"score={batch_shoreline.confidence:.3f}"
        )
        axis.axis("off")
    for axis in axes[len(batch_outputs):]:
        axis.axis("off")
    figure.tight_layout()
    plt.show()
    plt.close(figure)

    print(f"Successful: {len(batch_outputs)} | Failed: {len(batch_failures)}")
    for failure in batch_failures:
        print(failure)
else:
    print("Batch preview skipped. Set RUN_BATCH_PREVIEW=True to run it.")

## 11. Labeled validation on both locations

This is the decision cell before changing a global default. It runs all 6 labeled Entrance images and all 6 labeled Porch images. Every row is one image and every column is one `contrast_threshold`.

- **Green**: labeled water-mask boundary after registration.
- **Red**: predicted shoreline.
- `median error`: one-way median distance from prediction to the labeled boundary, in reference-image pixels.

The metric is useful for comparison, but the visual result still matters because annotation masks include occlusions and small internal boundaries.

In [ ]:
from yam_yabasha.registration import warp_image

RUN_LABELED_VALIDATION = False  # requires ignored data/*/images and data/*/masks
VALIDATION_THRESHOLDS = [-0.10, -0.05, 0.00, 0.05, 0.10]
VALIDATION_LOCATIONS = ("entrance", "porch")


def labeled_pairs(location):
    image_dir = ROOT / "data" / location / "images"
    mask_dir = ROOT / "data" / location / "masks"
    pairs = []
    for path in list_images(image_dir):
        task_number = int(path.stem.replace("task", ""))
        mask_path = mask_dir / f"task-{task_number:04d}.png"
        if mask_path.exists():
            pairs.append((path, mask_path))
    return pairs


def registered_ground_truth(mask_path, prediction):
    mask = load_image(mask_path, grayscale=True)
    registration = prediction.registration
    if registration is not None and registration.transform is not None:
        mask = warp_image(
            mask,
            registration.transform,
            prediction.aligned_image.shape[:2],
            config.registration.transform_type,
            interpolation=cv2.INTER_NEAREST,
        )
    return mask


def mask_boundary(mask):
    binary = (mask > 127).astype(np.uint8)
    kernel = np.ones((3, 3), dtype=np.uint8)
    return cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, kernel) > 0


def prediction_to_gt_error(points, gt_boundary):
    if len(points) == 0 or not np.any(gt_boundary):
        return np.nan
    distance_map = cv2.distanceTransform(
        (~gt_boundary).astype(np.uint8),
        cv2.DIST_L2,
        5,
    )
    xs = np.clip(points[:, 0], 0, distance_map.shape[1] - 1)
    ys = np.clip(points[:, 1], 0, distance_map.shape[0] - 1)
    return float(np.median(distance_map[ys, xs]))


if RUN_LABELED_VALIDATION:
    if "labeled_prediction_cache" not in globals():
        labeled_prediction_cache = {}
    validation_rows = []

    for validation_location in VALIDATION_LOCATIONS:
        pairs = labeled_pairs(validation_location)
        validation_hint = load_hint(
            ROOT / "data" / validation_location / "shoreline_hint.json"
        )
        figure, axes = plt.subplots(
            len(pairs),
            len(VALIDATION_THRESHOLDS),
            figsize=(4 * len(VALIDATION_THRESHOLDS), 3.2 * len(pairs)),
            dpi=90,
        )
        axes = np.atleast_2d(axes)

        for row_index, (validation_image, mask_path) in enumerate(pairs):
            cache_key = (
                validation_location,
                str(validation_image),
                WATER_PROMPTS,
                LAND_PROMPTS,
                WATER_COMBINE,
                LAND_COMBINE,
            )
            if cache_key not in labeled_prediction_cache:
                print(f"Running CLIPSeg: {validation_location}/{validation_image.name}")
                labeled_prediction_cache[cache_key] = pipeline.process(
                    validation_image,
                    validation_location,
                    register=True,
                    save_outputs=False,
                )
            validation_prediction = labeled_prediction_cache[cache_key]
            aligned_gt_mask = registered_ground_truth(mask_path, validation_prediction)
            full_gt_boundary = mask_boundary(aligned_gt_mask)

            for column_index, threshold in enumerate(VALIDATION_THRESHOLDS):
                validation_config = replace(
                    fast_config,
                    contrast_threshold=threshold,
                )
                validation_line = extract_shoreline(
                    validation_prediction.segmentation.water_probability,
                    validation_prediction.segmentation.land_probability,
                    validation_hint,
                    validation_config,
                )
                relevant_gt = full_gt_boundary & (validation_line.corridor_mask > 0)
                error_px = prediction_to_gt_error(validation_line.points, relevant_gt)

                overlay = validation_prediction.aligned_image.copy()
                thick_gt = cv2.dilate(relevant_gt.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
                overlay[thick_gt] = (0, 255, 0)
                overlay = draw_polyline(overlay, validation_line.points, (0, 0, 255), 4)

                axis = axes[row_index, column_index]
                axis.imshow(preview(rgb(overlay), max_side=650))
                error_label = "n/a" if not np.isfinite(error_px) else f"{error_px:.1f}px"
                axis.set_title(
                    f"{validation_image.stem} | t={threshold:+.2f}\n"
                    f"median error={error_label}"
                )
                axis.axis("off")
                validation_rows.append(
                    {
                        "location": validation_location,
                        "image": validation_image.stem,
                        "threshold": threshold,
                        "median_error_px": error_px,
                        "prediction_points": len(validation_line.points),
                        "candidates": len(validation_line.candidates),
                    }
                )

        figure.suptitle(
            f"{validation_location.title()}: green label vs red prediction",
            fontsize=16,
        )
        figure.tight_layout()
        plt.show()
        plt.close(figure)

    validation_table = pd.DataFrame(validation_rows)
    validation_summary = (
        validation_table.groupby(["location", "threshold"], as_index=False)
        .agg(
            median_error_px=("median_error_px", "median"),
            mean_error_px=("median_error_px", "mean"),
            images_with_prediction=("prediction_points", lambda values: int((values > 0).sum())),
        )
        .sort_values(["location", "median_error_px"])
    )
    display(validation_summary.round(2))
else:
    print("Labeled validation skipped.")